In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import os
import re
import matplotlib.pyplot as plt
import scipy as sc
import cartopy.crs as ccrs
# import cartopy.feature as cfeature
import calendar
import pickle
from minisom import MiniSom
from sklearn.preprocessing import MinMaxScaler, scale
%load_ext autoreload
%autoreload 2

# Preparación de datos

## Filtro ondas Rossby

## Filtro ondas Kelvin

## Filtro ondas del este

# Extraccion de patrones SOM

In [ ]:
var_scale = dict()
scalers = dict()

#Extraccion dimensiones
time,lat,lon = var_som['agua_ptable'].shape

# Inicializar diccionario para almacenar los scalers
scalers = dict()

# Inicializar lista para almacenar las variables escaladas
scaled_vars = list()

# Iterar sobre cada variable en var_som
for var_name, data in var_som.items():
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(data.reshape(time, -1))  # Ajustar y transformar
    scalers[var_name] = scaler  # Guardar el scaler para desescalado futuro
    scaled_vars.append(data_scaled)  # Almacenar los datos escalados

# Concatenar todas las variables escaladas a lo largo de la última dimensión
X_som = np.hstack(scaled_vars)  # Dimensión final (16071, 108*133*4)

## Modelo

In [ ]:

# Definir el tamaño del SOM
som_x, som_y = 4,4
X_som  # Datos ya escalados

# Calcular la hipotenusa del SOM
hypotenuse = np.sqrt(som_x**2 + som_y**2)

# Definir hiperparámetros iniciales
alpha_0 = 0.5  # Learning rate inicial
sigma_0 = 2.0  # Sigma inicial
num_iterations = 500  # Iteraciones de entrenamiento

# Inicializar el SOM
som = MiniSom(som_x, som_y, X_som.shape[1], 
              sigma=sigma_0, learning_rate=alpha_0, 
              neighborhood_function='gaussian')

som.random_weights_init(X_som)

# Listas para almacenar errores
q_error = []
t_error = []

# Entrenamiento aleatorio con actualización progresiva
for t in range(1, num_iterations + 1):
    alpha_t = alpha_0 / (1 + t / num_iterations)  # Decaimiento de alpha
    sigma_t = sigma_0 / (1 + t / (num_iterations / (hypotenuse - 1)))  # Decaimiento de sigma
    # print(sigma_t)

    # Asignar los valores actualizados al SOM manualmente
    # En caso que esto no funcione quitar
    som.learning_rate = alpha_t
    som.sigma = sigma_t

    # Seleccionar muestra aleatoria
    rand_i = np.random.randint(len(X_som))
    sample = X_som[rand_i]
    
    # Encontrar la mejor unidad de correspondencia (BMU)
    bmu = som.winner(sample)
    
    # Actualizar SOM con la muestra aleatoria
    som.update(sample, bmu, t, num_iterations)  
    
    # Guardar errores
    q_error.append(som.quantization_error(X_som))
    t_error.append(som.topographic_error(X_som))

    # if t%10 == 0:
    print(f'Iteracion:{t}')

## Extracción de resultados

In [ ]:
# Gráficos de error
path = r'G:\Mi unidad\Maestria_JulianToro\Figuras\Figuras_SOM\RMM_anomalias'
plt.figure(figsize=(10, 4))
plt.subplot(2,1,1)
plt.plot(range(num_iterations), q_error, label="Quantization Error")
plt.ylabel("Error de Cuantización")
plt.grid()
plt.legend()

plt.subplot(2,1,2)
plt.plot(range(num_iterations), t_error, label="Topographic Error")
plt.ylabel("Error Topográfico")
plt.legend()
plt.xlabel("Iteraciones")
plt.grid()
name = 'errores_4_4_SON.png'
name_save = os.path.join(path,name)
plt.savefig(name_save)


In [ ]:
path_guardado = r'G:\Mi unidad\Maestria_JulianToro\codigos\SOM'
# Guardar resultados del Win map
win_map = som.win_map(X_som)
# Guardar diccionario con los patrones de cada nodo

file_name = "som_win_map_4x4_SON.pkl"
path_save = os.path.join(path_guardado,file_name)
with open(path_save, "wb") as f:
    pickle.dump(win_map, f)

In [ ]:
# Obtener los nodos ganadores para cada muestra en X_som
winners = np.array([som.winner(x) for x in X_som])

# Guardar los nodos ganadores
winner_name = "som_winners_4_4_SON.npy"
path_save = os.path.join(path_guardado,winner_name)
np.save(path_save, winners)

In [ ]:
# Cargar los patrones asignados a cada nodo
path_save = r"G:\Mi unidad\Maestria_JulianToro\codigos\SOM\som_win_map_4_4.pkl"
with open(path_save, "rb") as f:
    win_map_v2 = pickle.load(f)

In [ ]:
path_save = r"G:\Mi unidad\Maestria_JulianToro\codigos\SOM\som_winners_4_4.npy"
winners = np.load(path_save)

In [ ]:
original_data = {key:data.reshape(time,-1) for key,data in var_som.items() }

In [ ]:
from collections import defaultdict

# === Reconstrucción del arreglo completo ===
X_reconstructed = np.zeros_like(X_som)
index = 0
for node, patterns in win_map_v2.items():
    num_patterns = len(patterns)
    X_reconstructed[index:index + num_patterns] = patterns
    index += num_patterns

print('Ya se genero X_reconstruido...')
# === Separar las variables de X_reconstructed ===
var_shapes = (108, 133)  # Dimensiones espaciales
split_indices = np.cumsum([original_data[v].shape[1] for v in original_data])[:-1]
X_vars = np.split(X_reconstructed, split_indices, axis=1)
print('Ya se separaron las variables')
# === Desescalar cada variable ===
X_descaled = {}
for i, var_name in enumerate(original_data.keys()):
    # Primero genera las variables de escalamiento
    data = var_som[var_name]
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(data.reshape(time, -1))
    # Luego desescala los datos
    X_descaled[var_name] = scaler.inverse_transform(X_vars[i])  # Desescalado

print('Ya se desescalaron los datos')
# === Reorganizar por nodo ===
node_patterns_descaled = defaultdict(lambda: defaultdict(list))
index = 0
for node, patterns in win_map_v2.items():
    num_patterns = len(patterns)
    for i, var_name in enumerate(original_data.keys()):
        node_patterns_descaled[node][var_name] = X_descaled[var_name][index:index + num_patterns]
    index += num_patterns

print('Ya se asociaron los datos a los nodos')

del original_data

In [ ]:
lista_nodos = sorted(list(node_patterns_descaled.keys()))
# Diccionario que relaciona los nodos con lo puntos de grafico
dict_nodos = {v:k for k,v in enumerate(lista_nodos)}

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Definir nombres de variables y dimensiones espaciales
variables = list(var_som.keys())

unidades = {'agua_ptable': '$kgm^{-2}$', 
            'div_transporte': '$Kgm^{-2}s^{-1}$', 
            'olr':'$W/m^2$',
            'div_200':'$1/s$',}

# Definir dimensiones espaciales
lon = np.arange(-90,-56.75,0.25)
lat = np.arange(-5.875,20.90,0.25)
# lat = ds.lat.values
# lon = ds.lon.values
dim_lat = len(lat)
dim_lon = len(lon)


var_shapes = (dim_lat, dim_lon)  # Ahora está definido correctamente

# Ruta para guardar las figuras
path = r'G:\Mi unidad\Maestria_JulianToro\Figuras\Figuras_SOM\resultados_EC'
x,y = 4,4

# Iterar sobre cada variable
for var_name in variables:
    print(f"Generando gráficos para {var_name}...")

    # 1. Determinar los valores mínimo y máximo para normalizar la barra de color
    vmin = min(np.mean(node_data[var_name], axis=0).min() for node_data in node_patterns_descaled.values())
    vmax = max(np.mean(node_data[var_name], axis=0).max() for node_data in node_patterns_descaled.values())

    # Rangos simetricos
    max_value = np.max([np.abs(vmin),vmax])

    # 2. Crear la figura y subgráficos con proyección geográfica
    fig, axes = plt.subplots(x, y, figsize=(15, 15), 
                             subplot_kw={'projection': ccrs.PlateCarree()},
                             sharey=True,)
    axes = axes.flatten()

    # 3. Graficar cada nodo con características geográficas
    print('Empece a graficar')
    for node, node_data in node_patterns_descaled.items():
        idx = dict_nodos[node]
        ax = axes[idx]
        data_plot = np.mean(node_data[var_name], axis=0).reshape(dim_lat,dim_lon)

        # Agregar el mapa de colores
        lon2D, lat2D = np.meshgrid(lon, lat)
        mesh = ax.pcolormesh(lon, lat, data_plot, cmap='RdBu',
                             vmin=-max_value, vmax=max_value,  
                         transform=ccrs.PlateCarree())

        # mesh = ax.pcolormesh(lon, lat, data_plot, cmap='RdBu',
        #                      vmin=vmin, vmax=vmax, 
        #                      transform=ccrs.PlateCarree(),
        #                       shading = 'auto')

        # Agregar líneas de costas, fronteras de países y límites de estados
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor='black')
        ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.5)
        ax.add_feature(cfeature.LAKES, edgecolor='black', facecolor='none')

        gl = ax.gridlines(draw_labels=True, 
                          linestyle="--", 
                          linewidth=0.3, alpha=0.7)
        
        gl.right_labels = False
        gl.top_labels = False

        if idx % 4 == 0: # Si es la primera columna (0, 4, 8)
            gl.left_labels = True
        else:
            gl.left_labels = False

        if idx >= 12: # Si es la última fila (
            gl.bottom_labels = True
        else:
            gl.bottom_labels = False
        # Tamaño de las etiquetas
        gl.xlabel_style = {'size': 13,
                           'rotation': 90} 
        gl.ylabel_style = {'size': 13}  
        
        # Configurar etiquetas
        ax.set_title(f"Nodo {node}", 
                     fontsize=15,
                     weight='bold')

    # Agregar espacio entre subplots
    fig.subplots_adjust(hspace=0.3, wspace=0.2)
    # 4. Agregar una barra de color universal
    cbar = fig.colorbar(mesh, ax=axes, 
                        orientation='horizontal', 
                        fraction=0.05, pad=0.1)
    
    unidad = unidades[var_name]
    cbar.set_label(f"{unidad}",fontsize=15,)
    cbar.ax.tick_params(labelsize=15) 

    # Guardar la figura correctamente
    nombre_guardado = os.path.join(path, f"patrones_{var_name}_v2.png")
    plt.savefig(nombre_guardado, dpi=300, bbox_inches ='tight' )
    plt.close()